# Week 10 extension — extend conditioning + train experiments (notebook 10d)

In Week 10 the conditional diffusion model barely tied the classical
baseline on hard NLL. The 4-D conditioning we fed it
(`area_smoothed`, `mu_universal`, `model_sigma`, `amplitude`) is the *same*
information the classical parametric model already used to produce
`hist_par` — so the diffusion was being told what the subtraction already
absorbed. The goal of this notebook (and its evaluation sibling 10e) is
to find out how much performance reverse diffusion can give us when we
**(a)** feed it conditioning the parametric model didn't already absorb
and **(b)** inject it through a stronger mechanism than raw concat.

This is a *staged* ablation study. You will progress through the
experiment menu one knob at a time — never enabling two new experiments
in the same session — so any NLL change in 10e is attributable to a
single cause. Encouragement here is for *lots* of experimentation,
disciplined.

The notebook has four parts:

1. **Part A — Build the augmented parquet** (Tasks 60–63). Add the new
   conditioning columns (cycle/hemisphere id, opposite-hemisphere
   summaries, smoothed-area trajectory) and write
   `diffusion_windows_v2.parquet` next to this notebook. **Rebuilt every
   run**, never cached, so any change to your augmentation logic
   propagates immediately.
2. **Part B — wandb setup + experiment menu** (Tasks 64–65). Each
   student gets their own wandb project; the experiment menu lists eight
   variants you'll progress through.
3. **Part C — Disciplined sweep** (Task 66). Enable a subset of
   experiments, train each, save a checkpoint per variant.
4. **Part D — Visual sanity check** of the most recent training.

> The **test split is reserved for the PI**. Every cell in this
> notebook filters to `split in {"train", "val"}`. Do not change that.


In [ ]:
# Standard Week 10 setup: locate the repo, install if missing.

import os, subprocess, sys

# Keep this path if working in Colab
# repo_path = "/content/butterflai"

# Use this path if working locally
repo_path = "../../"

if not os.path.isdir(repo_path):
    subprocess.run(["git", "clone", "https://github.com/SwRI-IDEA-Lab/butterflai.git", repo_path], check=True)
else:
    try:
        subprocess.run(["git", "-C", repo_path, "pull"], check=True)
    except subprocess.CalledProcessError as e:
        print(f"git pull skipped: {e}")
sys.path.insert(0, repo_path)
from infrastructure.utils.colab_setup import setup
setup()


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import torch
from einops import repeat

import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger, CSVLogger
import wandb


In [ ]:
# Standard Week-10 setup: bootstrap sys.path and locate artifacts.
import os, sys

# Bootstrap: enough sys.path entries to import the helper. The helper
# itself adds the repo root + remaining week_08/09/10 dirs.
for _p in [os.path.abspath(os.path.join(repo_path, "weeks", "week_10")),
           os.path.abspath(os.path.join(repo_path, "weeks", "week_09"))]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from conditioned_infrastructure import find_week10_artifacts
paths = find_week10_artifacts()
print(f"using conditioned_infrastructure from: {paths['conditioned_py']}")

from unconditioned_infrastructure import make_cosine_schedule
from conditioned_infrastructure import (
    ExtendedConditionalResidualDataset,
    train_experiment,
    load_trained_experiment,
    sample_conditional_extended,
)
from butterflAI_model import ButterflAIModel

_WEEK10_DIR = paths["week10_dir"]
PARQUET_V2  = os.path.join(_WEEK10_DIR, "diffusion_windows_v2.parquet")
CKPT_DIR    = _WEEK10_DIR

classical   = ButterflAIModel(paths["classical_weights"])

LAT_BINS    = np.linspace(0, 45, 16)
BIN_WIDTH   = 3.0
BIN_CENTERS = 0.5 * (LAT_BINS[:-1] + LAT_BINS[1:])

T = 200
alpha_np, sigma_np, _ = make_cosine_schedule(T=T, s=0.008)

# Load the v1 parquet — we extend it but never modify it.
# Test split is reserved for the PI.
windows_v1 = pd.read_parquet(paths["parquet_v1"])
windows_v1 = windows_v1.loc[windows_v1["split"].isin(["train", "val"])].reset_index(drop=True)
print(f"v1 parquet (train+val only): {len(windows_v1)} rows")
print(f"  splits: {windows_v1['split'].value_counts().sort_index().to_dict()}")
print(f"  cycles: {sorted(windows_v1['cycle'].unique())}")
print(f"v2 parquet target: {PARQUET_V2}")

---
## Part A — Build the augmented parquet

We're going to add three families of new conditioning columns to the
v1 parquet:

1. **Cycle and hemisphere identifiers** — a normalized cycle number and
   a north/south indicator.
2. **Opposite-hemisphere summaries** — for each window, the
   *contemporaneous* opposite-hemisphere activity. This is not leakage:
   contemporaneous opposite-hemisphere activity is operationally
   observable (an operational forecaster on the day of the same window
   would have it).
3. **Smoothed-area trajectory** — the past `K` windows of `area_smoothed`
   for the same hemicycle, as a short autoregressive history.

The build cells **rewrite `diffusion_windows_v2.parquet` every run**. We
deliberately do *not* gate on file existence — if you change how a
column is computed and don't see the change downstream, the most
common explanation is "the file was cached." We avoid that failure
mode by always rewriting.


---
## Task 60 — Cycle and hemisphere identifiers

Add two new columns to the dataframe:

- `cycle_norm`: cycle number rescaled to roughly `[-1, +1]` over the
  train range. The point of normalization is not to be exactly in
  `[-1, +1]` — it's to put the input on the same numerical scale as the
  other conditioning vectors so the network doesn't have to learn an
  outsized weight for it.
- `hemi_id`: `+1` for north, `-1` for south.

Both are cheap and let downstream experiments test whether
*structural* per-cycle / per-hemisphere effects survive once amplitude
is controlled for.


In [ ]:
# Task 60 — add cycle_norm and hemi_id.

windows_aug = windows_v1.copy()

# TODO: compute cycle_norm. Use train-set cycle range so this is well
# defined for both splits. Aim for the train cycle range to map roughly
# onto [-1, +1].
_train_cycles = windows_aug.loc[windows_aug["split"] == "train", "cycle"]
_cmin, _cmax  = _train_cycles.min(), _train_cycles.max()
windows_aug["cycle_norm"] = 2.0 * (windows_aug["cycle"] - _cmin) / (_cmax - _cmin) - 1.0

# TODO: compute hemi_id. +1 north, -1 south.
windows_aug["hemi_id"] = np.where(windows_aug["hemisphere"] == "north", 1.0, -1.0)

print("cycle_norm range:", windows_aug["cycle_norm"].min(), windows_aug["cycle_norm"].max())
print("hemi_id values  :", windows_aug["hemi_id"].unique())


---
## Task 61 — Opposite-hemisphere conditioning

For each window in hemisphere *h* at time `tau_center`, attach the
contemporaneous opposite-hemisphere activity summary:

- `opp_area_smoothed` — opposite hemisphere's `area_smoothed`
- `opp_mu_universal`  — opposite hemisphere's `mu_universal`
- `opp_amplitude`     — opposite hemisphere's `amplitude`
- `opp_valid`         — 1 if a matching opposite row was found at the
  same `(cycle, tau_center)`, else 0.

When `opp_valid == 0` (no matching opposite row), impute with the
**train-set mean** of each opposite-* column. This way the network always
sees a defined input; downstream you can decide whether to gate on the
mask.

**Implementation hint:** the cleanest way is a self-merge of the
dataframe with itself: produce a "right side" with `hemisphere`
flipped and renamed columns, then merge on `(cycle, tau_center)`.


In [ ]:
# Task 61 — attach contemporaneous opposite-hemisphere summaries.

# TODO: build a "right side" dataframe with hemisphere flipped and the
#       three columns we want renamed with the "opp_" prefix.
_flip = {"north": "south", "south": "north"}
_right = (windows_aug
          .loc[:, ["cycle", "tau_center", "hemisphere",
                   "area_smoothed", "mu_universal", "amplitude"]]
          .assign(opp_of_hemisphere=lambda d: d["hemisphere"].map(_flip))
          .drop(columns=["hemisphere"])
          .rename(columns={
              "area_smoothed": "opp_area_smoothed",
              "mu_universal":  "opp_mu_universal",
              "amplitude":     "opp_amplitude",
              "opp_of_hemisphere": "hemisphere",
          }))

# TODO: merge on (cycle, tau_center, hemisphere) so each row gets the
#       opposite-side row that has the *flipped* hemisphere stored under
#       the same hemisphere key.
windows_aug = windows_aug.merge(
    _right, on=["cycle", "tau_center", "hemisphere"],
    how="left", indicator="_opp_match",
)
windows_aug["opp_valid"] = (windows_aug["_opp_match"] == "both").astype(np.float32)
windows_aug = windows_aug.drop(columns=["_opp_match"])

# TODO: impute missing opp_* values with train-set means.
_opp_cols = ["opp_area_smoothed", "opp_mu_universal", "opp_amplitude"]
_train_mask = (windows_aug["split"] == "train") & (windows_aug["opp_valid"] == 1.0)
for _c in _opp_cols:
    _mean = windows_aug.loc[_train_mask, _c].mean()
    windows_aug[_c] = windows_aug[_c].fillna(_mean)

print(f"opp_valid coverage: {windows_aug['opp_valid'].mean():.3f}")
print(f"opp_area_smoothed (train, valid): "
      f"mean={windows_aug.loc[_train_mask, 'opp_area_smoothed'].mean():.3f}, "
      f"std={windows_aug.loc[_train_mask, 'opp_area_smoothed'].std():.3f}")


---
## Task 62 — Smoothed-area trajectory (cycle history)

For each window, attach the previous `K` values of `area_smoothed`
from the same hemicycle (same `cycle` AND same `hemisphere`), ordered
chronologically by `tau_center`. Columns: `area_lag1`, `area_lag2`, …,
`area_lagK`.

`area_lag1` is the *previous* window's smoothed area; `area_lagK` is
the one K steps back. Boundary windows (near the start of a hemicycle,
where fewer than K prior windows exist) get train-set-mean imputation
and a `traj_valid` column set to 0 for that row.

The point: cycle *history* is information that amplitude alone misses.
A window 6 months into a strong cycle and a window 6 months from the
end of a strong cycle have similar amplitude but very different
trajectory.

**Implementation hint:** sort within each hemicycle by `tau_center`,
then shift the `area_smoothed` series by 1, 2, …, K.


In [ ]:
# Task 62 — attach the smoothed-area trajectory (K lagged values).

K_LAGS = 4

# TODO: within each (cycle, hemisphere) group, sort by tau_center and
#       produce K columns of shifted area_smoothed.
_sorted = windows_aug.sort_values(["cycle", "hemisphere", "tau_center"])

_lag_cols = [f"area_lag{k}" for k in range(1, K_LAGS + 1)]
for k, col in enumerate(_lag_cols, start=1):
    _sorted[col] = _sorted.groupby(["cycle", "hemisphere"])["area_smoothed"].shift(k)

# A row is traj_valid iff *all* K lags exist.
_sorted["traj_valid"] = (~_sorted[_lag_cols].isna().any(axis=1)).astype(np.float32)

# TODO: impute boundary NaNs with train-set means.
_train_mask = (_sorted["split"] == "train") & (_sorted["traj_valid"] == 1.0)
for col in _lag_cols:
    _mean = _sorted.loc[_train_mask, col].mean()
    _sorted[col] = _sorted[col].fillna(_mean)

windows_aug = _sorted.sort_index()  # restore original row order

print(f"traj_valid coverage: {windows_aug['traj_valid'].mean():.3f}")
print(f"lag columns: {_lag_cols}")


---
## Task 63 — Write `diffusion_windows_v2.parquet` and sanity-check

Write the augmented dataframe. Sanity checks before we trust it
downstream:

- Row count unchanged from v1 (we did not gain or lose any windows).
- Every original v1 column is preserved bit-for-bit.
- New cond columns are finite **wherever the validity mask says they
  should be**.
- `split` column is unchanged.


In [ ]:
# Task 63 — write v2 + sanity checks.

# Sanity 1: row count.
assert len(windows_aug) == len(windows_v1), \
    f"row count drifted: {len(windows_aug)} vs v1 {len(windows_v1)}"

# Sanity 2: every v1 column preserved bit-for-bit.
for c in windows_v1.columns:
    assert c in windows_aug.columns, f"missing v1 column {c}"
    if windows_v1[c].dtype.kind in "fc":
        assert np.allclose(windows_v1[c].to_numpy(),
                           windows_aug[c].to_numpy(), equal_nan=True), c
    else:
        assert (windows_v1[c].astype(str).to_numpy()
                == windows_aug[c].astype(str).to_numpy()).all(), c

# Sanity 3: new columns finite where the validity masks allow.
NEW_COND_COLS = ["cycle_norm", "hemi_id",
                 "opp_area_smoothed", "opp_mu_universal", "opp_amplitude",
                 *[f"area_lag{k}" for k in range(1, K_LAGS + 1)]]
for c in NEW_COND_COLS:
    assert windows_aug[c].notna().all(), f"NaN remains in {c}"

# Sanity 4: split column unchanged.
assert (windows_aug["split"].to_numpy() == windows_v1["split"].to_numpy()).all()

# Always rewrite. Never cache.
windows_aug.to_parquet(PARQUET_V2, index=False)
print(f"wrote {PARQUET_V2}  ({len(windows_aug)} rows, {len(windows_aug.columns)} cols)")
print(f"new cond columns: {NEW_COND_COLS}")


---
## Part B — wandb setup and the experiment menu

### Task 64 — Per-student wandb project

Each student gets their **own** wandb project. Replace the placeholder
strings below with your handle. Every training run in this notebook
logs to that project with the experiment ID as the run name; you can
compare all your variants on a single dashboard.

If wandb is unavailable in your environment, training will fall back to
a local CSV logger automatically.


In [ ]:
# Task 64 — wandb identity. EDIT THESE.

WANDB_PROJECT = "butterflai-w10ext-<your-handle>"
WANDB_ENTITY  = "<your-wandb-username>"      # set to None if you don't use teams

assert "your-handle" not in WANDB_PROJECT, \
    "Set WANDB_PROJECT to your own project name before training."


### Task 65 — Design your own experiments

The Week 10 baseline (E0) reproduces the existing conditional
diffusion on the v2 parquet — no new knobs. Everything beyond it is
your call. Each variant you propose should change **one knob** from
the previous run and answer **one question**.

The knobs available are:

- **Cond groups** (`groups` / `consumed_keys`): `base`, plus any of
  `cyclehemi`, `opp`, `traj`.
- **Architecture** (`arch`): `concat` or `film`.
- **Classifier-free guidance**: `cond_dropout_p=0.1` at training time;
  10e sweeps the guidance weight at sampling.
- **Fourier lifting**: `fourier=True` lifts cond scalars via sin/cos.

The menu below escalates roughly by effort-per-insight. Pick what's
interesting, add a new entry to `EXPERIMENTS`, and progress one
variant per session.

**Level 1 — same cond, change the channel.**
Add one of the new cond groups (`cyclehemi`, `opp`, `traj`) to E0's
`consumed_keys`. *Does the diffusion's val NLL drop when given more
information, with the architecture held fixed?*

**Level 2 — same information, change the mechanism.**
Switch `arch` from `concat` to `film` while keeping `cond_base` only.
*Does the modulation mechanism alone close the gap with classical?*

**Level 3 — best information × best mechanism.**
Combine your best Level 1 cond set with FiLM. *Is the combined gain
additive, or did Level 2 already capture it?*

**Level 4 — guidance.**
Set `cond_dropout_p=0.1` and train. Sampling guidance is swept in 10e.
*Can sharpening the conditional density buy you margin over Level 3?*

**Level 5 — Fourier lifting.**
Set `fourier=True`. *Does sin/cos lifting of the cond scalars help the
network represent boundaries?*

You can go further — bump `hidden_dim` / `n_layers`, raise `K_LAGS`
back in Task 62, pair lagged opposite-hemisphere with trajectory, or
anything else you can defend. Different students should diverge here;
results pool in 10e.

In [ ]:
# Task 65 — experiment specs. Start with the baseline; add new entries
# below as you escalate (see the markdown above). See
# conditioned_infrastructure.build_model for the recognized keys.

_BASE_TEMPLATE = {
    "arch":           "concat",
    "consumed_keys":  ["cond_base"],
    "groups":         ["base"],
    "hidden_dim":     128,
    "n_layers":       3,
    "fourier":        False,
    "cond_dropout_p": 0.0,
    "max_epochs":     20000,
    "lr":             1e-3,
    "batch_size":     64,
}

def _spec(**overrides):
    d = dict(_BASE_TEMPLATE); d.update(overrides); return d

EXPERIMENTS = {
    "E0": _spec(),   # baseline — same 4-D cond on the v2 parquet
    # Add your own variants below, e.g.:
    #   "E1": _spec(consumed_keys=["cond_base", "cond_opp"],
    #               groups=["base", "opp"]),
    #   "E2": _spec(arch="film"),
}

for name, cfg in EXPERIMENTS.items():
    print(f"{name}: arch={cfg['arch']:6s}  consumed={cfg['consumed_keys']}  "
          f"fourier={cfg['fourier']}  cond_dropout_p={cfg['cond_dropout_p']}")

---
## Part C — Disciplined sweep

### Task 66 — Enable a subset and train

Discipline:

- Add at most **one new experiment per session** beyond the baseline.
  Two-knob-at-a-time changes make the 10e diff impossible to read.
- Each enabled experiment logs to your wandb project under its name
  (`E0`, `E1`, …); compare them on a single dashboard.
- The loop skips checkpoints that already exist on disk, so re-running
  the notebook does not retrain unless you delete the file.

In [ ]:
# Task 66 — enable, then train. EDIT THIS LIST.

ENABLED_EXPERIMENTS = ["E0"]   # add one experiment per session

for _name in ENABLED_EXPERIMENTS:
    if _name not in EXPERIMENTS:
        raise KeyError(f"unknown experiment {_name!r}; defined: {list(EXPERIMENTS)}")
    train_experiment(
        name=_name, cfg=EXPERIMENTS[_name],
        windows_aug=windows_aug, ckpt_dir=CKPT_DIR,
        wandb_project=WANDB_PROJECT, wandb_entity=WANDB_ENTITY,
        alpha_np=alpha_np, sigma_np=sigma_np, T=T,
        bin_centers=BIN_CENTERS, bin_width=BIN_WIDTH,
    )

---
## Part D — Visual sanity check on the most recent training

Sample a small batch of validation conditioning vectors and overlay
the diffusion's generated residuals against the ground truth. This is
a "did training collapse?" check — not a quantitative comparison. The
real evaluation lives in 10e.


In [ ]:
# Part D — quick overlay for the most recently trained checkpoint.

if not ENABLED_EXPERIMENTS:
    print("No experiments were trained this session — nothing to visualize.")
else:
    _name = ENABLED_EXPERIMENTS[-1]
    _cfg  = EXPERIMENTS[_name]
    lit, _, val_ds, _ = load_trained_experiment(
        _name, _cfg, windows_aug, CKPT_DIR, alpha_np, sigma_np,
    )

    n_show = 4
    cond_concat = torch.cat(
        [torch.stack([val_ds[i][k] for i in range(n_show)])
         for k in _cfg["consumed_keys"]],
        dim=-1,
    )
    truth = torch.stack([val_ds[i]["r_clean"] for i in range(n_show)]).numpy()
    truth_phys = truth * val_ds.bin_stds.numpy() + val_ds.bin_means.numpy()
    samples = sample_conditional_extended(lit, cond_concat, guidance_w=0.0).cpu().numpy()

    fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 3.2), sharey=True)
    for i, ax in enumerate(axes):
        w = BIN_WIDTH * 0.4
        ax.bar(BIN_CENTERS - w / 2, truth_phys[i], width=w, color="C0", label="truth")
        ax.bar(BIN_CENTERS + w / 2, samples[i],    width=w, color="C2", label="sampled")
        ax.axhline(0, color="k", lw=0.4)
        ax.set_title(f"val window {i}")
        ax.set_xlabel("|latitude| (°)")
    axes[0].set_ylabel("residual"); axes[0].legend()
    fig.suptitle(f"{_name}: visual sanity check")
    fig.tight_layout(); plt.show()

---
## Handoff to `10e_diffusion_NLL_ablations.ipynb`

Each enabled experiment has produced a checkpoint named
`ckpt_E{N}.ckpt` in this directory. Notebook 10e will:

- Train a small **oracle MLP** per experiment (cond → Gaussian residual
  params): the resulting NLL is an *upper bound* on what any diffusion
  with that cond set could achieve.
- Score every `ckpt_E*.ckpt` on the val split (K=100 conditional
  samples per window) and produce the headline NLL bar chart with the
  classical baseline and Week 10's existing checkpoint as anchors.
- Run the guidance sweep for any E6 checkpoint.

**The test split is reserved for the PI.** Do not modify the
`split.isin(["train", "val"])` filter at the top of this notebook, and
do not introduce one in 10e.
